In [1]:
!pip install xgboost


In [2]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report
import xgboost as xgb
import datetime

In [3]:
# ---------------- Setup Paths ----------------
DATA_PATH = "../data"
MODELS_PATH = "../models"
os.makedirs(MODELS_PATH, exist_ok=True)

TARGET_COL = "isFraud"
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [4]:
# ---------------- Load Data ----------------
try:
    train_df = pd.read_csv(os.path.join(DATA_PATH, "train_final_processed.csv"))
except FileNotFoundError:
    print("Error: train_final_processed.csv not found.")
    exit()

# Separate features and target
X = train_df.drop(columns=[TARGET_COL, "TransactionID"])
y = train_df[TARGET_COL]

In [5]:
# Drop intermediate time columns if any exist
cols_to_drop_dt = [c for c in X.columns if c.startswith("TransactionDT") or c.startswith("Transaction_day")]
X.drop(columns=cols_to_drop_dt, errors="ignore", inplace=True)

print("Loaded {} features.".format(len(X.columns)))
print("X shape: {}, y shape: {}".format(X.shape, y.shape))

# ---------------- Class Imbalance ----------------
pos_weight = (y == 0).sum() / (y == 1).sum()
print("Class Imbalance Ratio (Non-Fraud / Fraud): {:.2f}".format(pos_weight))

Loaded 341 features.
X shape: (590540, 341), y shape: (590540,)
Class Imbalance Ratio (Non-Fraud / Fraud): 27.58


In [7]:
# ---------------- XGBoost Parameters ----------------
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": 0.05,
    "n_estimators": 1000,
    "max_depth": 12,
    "colsample_bytree": 0.8,
    "subsample": 0.8,
    "lambda": 1,
    "alpha": 0,
    "scale_pos_weight": pos_weight,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "tree_method": "hist",
    "verbosity": 0,
    "early_stopping_rounds": 100
}

# ---------------- Cross Validation ----------------
N_SPLITS = 5
kf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_preds = np.zeros(X.shape[0])
models = []

print("\nStarting {}-Fold XGBoost Training...".format(N_SPLITS))
start_time = datetime.datetime.now()

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    models.append(model)

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print("Fold {}/{} | AUC: {:.6f}".format(fold + 1, N_SPLITS, fold_auc))

end_time = datetime.datetime.now()
print("\nTraining Complete. Total Time: {}".format(end_time - start_time))



Starting 5-Fold XGBoost Training...
Fold 1/5 | AUC: 0.967716
Fold 2/5 | AUC: 0.968669
Fold 3/5 | AUC: 0.968558
Fold 4/5 | AUC: 0.971080
Fold 5/5 | AUC: 0.970959

Training Complete. Total Time: 0:14:32.734086


In [10]:
# ---------------- Final Evaluation ----------------
overall_auc = roc_auc_score(y, oof_preds)
print("=========================================")
print("OVERALL OOF AUC: {:.6f}".format(overall_auc))
print("=========================================")

oof_labels = (oof_preds > 0.5).astype(int)
print("\nClassification Report (Threshold 0.5):")
print(classification_report(y, oof_labels))

OVERALL OOF AUC: 0.969376

Classification Report (Threshold 0.5):
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    569877
           1       0.90      0.72      0.80     20663

    accuracy                           0.99    590540
   macro avg       0.94      0.86      0.90    590540
weighted avg       0.99      0.99      0.99    590540



In [15]:
# ---------------- Save Final Model ----------------
# Retrain using average best iteration
best_iterations = [model.best_iteration for model in models]
final_n_estimators = int(np.mean(best_iterations))

# 1. Create a copy of the params to modify
final_xgb_params = xgb_params.copy() 

# 2. Remove 'n_estimators' and 'early_stopping_rounds' if they exist.
#    We remove them because:
#    - 'n_estimators' is set separately below. (Fix for previous TypeError)
#    - 'early_stopping_rounds' requires 'eval_set', which is not provided 
#      when fitting the final model on the whole dataset. (Fix for current ValueError)
if 'n_estimators' in final_xgb_params:
    del final_xgb_params['n_estimators']

# **NEW LINE ADDED HERE**
# Remove the parameter that enables Early Stopping
if 'early_stopping_rounds' in final_xgb_params: 
    del final_xgb_params['early_stopping_rounds']

# 3. Pass the cleaned dictionary and the final n_estimators
final_model = xgb.XGBClassifier(
    **final_xgb_params,  # Unpack the parameters without n_estimators or early_stopping_rounds
    n_estimators=final_n_estimators # Set the desired value explicitly
)

# This fit will now use final_n_estimators rounds without trying to use a missing eval_set
final_model.fit(X, y) 
joblib.dump(final_model, os.path.join(MODELS_PATH, "xgb_final_model.pkl"))
print("\nFinal XGBoost model saved.")


Final XGBoost model saved.


In [16]:
# ---------------- Feature Importance ----------------
avg_importance = np.mean(
    [model.feature_importances_ for model in models],
    axis=0
)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": avg_importance
}).sort_values(by="importance", ascending=False)

print("\nTop 20 Features by Importance:")
print(importance_df.head(20))


Top 20 Features by Importance:
    feature  importance
174    V258    0.196085
176     V70    0.123623
234    V294    0.037296
135     V91    0.037212
178    V201    0.033774
311    V218    0.022621
130    V257    0.014877
235     V90    0.012080
105     C14    0.008580
268      C8    0.007925
10      V69    0.006169
214      C3    0.005828
121    V283    0.005595
225   addr2    0.004104
33      V65    0.004075
245    V264    0.003923
306    V285    0.003766
83     V187    0.003710
12     V312    0.003550
321    V296    0.003445
